In [ ]:
from pathlib import Path
import json
import tifffile
import sys

sys.path.append(str(Path.cwd().parent))
import conf


IMAGE_DIR = conf.IMAGE_DIR
LABELS_PATH = conf.LABELS_PATH
ANNOTATION_PATH = conf.ANNOTATION_PATH

WSI_PAGES = conf.WSI_PAGES
CNN_INPUT_HEIGHT = conf.CNN_INPUT_HEIGHT
CNN_INPUT_WIDTH = conf.CNN_INPUT_WIDTH
MAX_CROP_DEPTH = conf.MAX_CROP_DEPTH

In [21]:
def matrix_dict_to_list(m):
    return [
        [m["t_00"], m["t_01"], m["t_02"]],
        [m["t_10"], m["t_11"], m["t_12"]],
        [m["t_20"], m["t_21"], m["t_22"]],
    ]


def load_pairs_from_labels(labels_path, image_dir):
    with open(labels_path, "r", encoding="utf-8") as f:
        labels = json.load(f)

    pairs = []

    for pair_id, item in enumerate(labels):
        source_id = item["source_image_id"]
        target_id = item["target_image_id"]

        source_path = image_dir / f"{source_id}.data"
        target_path = image_dir / f"{target_id}.data"

        if not source_path.exists():
            raise FileNotFoundError(source_path)

        if not target_path.exists():
            raise FileNotFoundError(target_path)

        pairs.append({
            "pair_id": pair_id,

            # source/moving gets registered to target/fixed
            "moving_path": str(source_path),
            "fixed_path": str(target_path),

            "source_image_id": source_id,
            "target_image_id": target_id,

            "registration_error": item["registration_error"],
            "transformation_matrix": matrix_dict_to_list(item["transformation_matrix"]),
        })

    return pairs

In [22]:
def choose_pair_pyramid_page(fixed_path, moving_path, crop_depth, input_height=CNN_INPUT_HEIGHT, input_width=CNN_INPUT_WIDTH):
    grid = 2 ** crop_depth

    with tifffile.TiffFile(fixed_path) as fixed_slide, tifffile.TiffFile(moving_path) as moving_slide:
        candidates = []

        for page_idx in WSI_PAGES:
            fixed_h, fixed_w = fixed_slide.pages[page_idx].shape[:2]
            moving_h, moving_w = moving_slide.pages[page_idx].shape[:2]

            fixed_tile_h = fixed_h // grid
            fixed_tile_w = fixed_w // grid

            moving_tile_h = moving_h // grid
            moving_tile_w = moving_w // grid

            min_tile_h = min(fixed_tile_h, moving_tile_h)
            min_tile_w = min(fixed_tile_w, moving_tile_w)

            if min_tile_h >= input_height and min_tile_w >= input_width:
                candidates.append((page_idx, min_tile_h, min_tile_w))

    if not candidates:
        return None

    return candidates[-1]

In [23]:
def build_quadtree_index(pairs, max_crop_depth, input_height=CNN_INPUT_HEIGHT, input_width=CNN_INPUT_WIDTH):
    tile_jobs = []

    for pair in pairs:
        for crop_depth in range(max_crop_depth + 1):
            chosen = choose_pair_pyramid_page(
                fixed_path=pair["fixed_path"],
                moving_path=pair["moving_path"],
                crop_depth=crop_depth,
                input_height=input_height,
                input_width=input_width,
            )

            if chosen is None:
                continue

            pyramid_page_idx, tile_h, tile_w = chosen
            grid = 2 ** crop_depth

            for y_idx in range(grid):
                for x_idx in range(grid):
                    tile_jobs.append({
                        "pair_id": pair["pair_id"],

                        "fixed_path": pair["fixed_path"],
                        "moving_path": pair["moving_path"],

                        "source_image_id": pair["source_image_id"],
                        "target_image_id": pair["target_image_id"],

                        "crop_depth": crop_depth,
                        "grid": grid,
                        "x_idx": x_idx,
                        "y_idx": y_idx,

                        "pyramid_page_idx": pyramid_page_idx,
                        "tile_h": tile_h,
                        "tile_w": tile_w,
                        "cnn_input_height": input_height,
                        "cnn_input_width": input_width,

                        "registration_error": pair["registration_error"],
                        "transformation_matrix": pair["transformation_matrix"],
                    })

    return tile_jobs


def save_quadtree_index(tile_jobs, annotation_path):
    annotation_path.parent.mkdir(parents=True, exist_ok=True)

    with open(annotation_path, "w", encoding="utf-8") as f:
        json.dump(tile_jobs, f, indent=2)

In [24]:
pairs = load_pairs_from_labels(LABELS_PATH, IMAGE_DIR)

tile_jobs = build_quadtree_index(
    pairs=pairs,
    max_crop_depth=MAX_CROP_DEPTH,
    input_height=CNN_INPUT_HEIGHT,
    input_width=CNN_INPUT_WIDTH,
)

save_quadtree_index(tile_jobs, ANNOTATION_PATH)

print(f"pairs: {len(pairs)}")
print(f"tile jobs: {len(tile_jobs)}")
print(f"saved to: {ANNOTATION_PATH}")

print("\nFirst job:")
print(tile_jobs[0])

print("\nLast job:")
print(tile_jobs[-1])

FileNotFoundError: [Errno 2] No such file or directory: 'setup/labels.json'